# YOLOv11 Jersey Number Classifier Training

## Before running in Kaggle

1. Turn **Internet** on.
2. Set **Accelerator** to **GPU T4** (or better).
3. Add your labeled crops dataset as a Kaggle input.
4. Run cells top-to-bottom.

## What this notebook does

- Loads a CSV of crop filenames + jersey number labels.
- Filters out rare classes and builds a stratified train/val split.
- Trains a **YOLOv11s-cls** image classifier with augmentations tuned for small jersey crop images.
- Saves  to  for download.

In [ ]:
# =========================
# 0. INSTALL PACKAGES
# =========================

!pip install ultralytics

In [ ]:
# =========================
# 1. IMPORTS + GPU CHECK
# =========================

import os
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found. In Kaggle settings, set Accelerator to GPU T4, then restart session.")

In [ ]:
# =========================
# 2. CONFIG
# =========================

# Paths
DATASET_DIR = Path("/kaggle/input/datasets/andrewleick/labels1")
OUTPUT_DIR  = Path("/kaggle/working/jersey_cls")
CSV_PATH    = DATASET_DIR / "labels_clean.csv"

# Core training settings
MODEL_NAME = "yolo11s-cls.pt"
RUN_NAME   = "jersey_ocr"

EPOCHS     = 100
BATCH_SIZE = 32
IMG_SIZE   = 224
PATIENCE   = 15

# Dataset filtering
MIN_CLASS_SAMPLES = 10   # drop classes with fewer samples than this
RANDOM_SEED       = 67

# Augmentation settings (tuned for small, variable-aspect jersey crops)
AUG_DROPOUT  = 0.3
AUG_DEGREES  = 15    # max rotation in degrees
AUG_SHEAR    = 5     # max shear in degrees
AUG_HSV_V    = 0.4   # value (brightness) jitter
AUG_HSV_S    = 0.4   # saturation jitter
# Flips are disabled — mirroring jersey numbers changes their meaning
AUG_FLIPLR   = 0.0
AUG_FLIPUD   = 0.0

In [ ]:
# =========================
# 3. BUILD CLASSIFICATION DATASET
# =========================

def build_classification_dataset():
    """
    Reads the label CSV, filters rare classes, does a stratified train/val split,
    and copies crops into the folder structure YOLO expects:
        OUTPUT_DIR/train/<label>/<filename>
        OUTPUT_DIR/val/<label>/<filename>
    """
    # Wipe any previous run's data
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    images_dir = DATASET_DIR / "crops" / "crops"

    # Reconcile CSV against disk
    csv_filenames  = set(df["filename"])
    disk_filenames = set(f.name for f in images_dir.iterdir() if f.is_file()) if images_dir.exists() else set()

    not_in_csv = disk_filenames - csv_filenames
    if not_in_csv:
        print(f"Skipping {len(not_in_csv)} crops with no CSV label.")
        df = df[df["filename"].isin(disk_filenames)]

    # Drop rare classes that can't be meaningfully learned or split
    label_counts = df["label"].value_counts()
    rare = label_counts[label_counts < MIN_CLASS_SAMPLES].index
    if len(rare) > 0:
        print(f"Dropping {len(rare)} classes with fewer than {MIN_CLASS_SAMPLES} images: {sorted(rare.tolist())}")
        df = df[~df["label"].isin(rare)]

    print(f"Dataset after filtering: {len(df)} images, {df['label'].nunique()} classes")

    # Stratified split
    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=RANDOM_SEED, stratify=df["label"]
    )

    # Auto-heal: drop any classes still missing from val after the split
    train_classes = set(train_df["label"].unique())
    val_classes   = set(val_df["label"].unique())
    if train_classes != val_classes:
        missing = train_classes - val_classes
        print(f"Warning: dropping {len(missing)} classes still missing from val: {sorted(missing)}")
        train_df = train_df[~train_df["label"].isin(missing)]
        val_df   = val_df[~val_df["label"].isin(missing)]

    print(f"Train: {len(train_df)}  Val: {len(val_df)}  Classes: {train_df['label'].nunique()}")

    for split, split_df in [("train", train_df), ("val", val_df)]:
        for _, row in split_df.iterrows():
            src = images_dir / row["filename"]
            dst = OUTPUT_DIR / split / str(row["label"]) / row["filename"]
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(str(src), str(dst))

    return str(OUTPUT_DIR)

data_dir = build_classification_dataset()
print("Dataset ready at:", data_dir)

In [ ]:
# =========================
# 4. INSPECT LABEL DISTRIBUTION
# =========================

df = pd.read_csv(CSV_PATH)
print("CSV columns:", df.columns.tolist())
print("Label value counts:")
print(df["label"].value_counts().to_string())

In [ ]:
# =========================
# 5. TRAIN
# =========================

def train(data_dir: str):
    model = YOLO(MODEL_NAME)
    model.train(
        data=data_dir,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        project="/kaggle/working/runs",
        name=RUN_NAME,
        patience=PATIENCE,

        # Regularization
        dropout=AUG_DROPOUT,

        # Spatial augmentations (safe for digits)
        degrees=AUG_DEGREES,
        shear=AUG_SHEAR,

        # Color augmentations
        hsv_v=AUG_HSV_V,
        hsv_s=AUG_HSV_S,

        # Flips disabled — mirroring changes number meaning
        fliplr=AUG_FLIPLR,
        flipud=AUG_FLIPUD,
    )
    print(f"Training complete! Best weights saved to: /kaggle/working/runs/{RUN_NAME}/weights/best.pt")

train(data_dir)

In [ ]:
# =========================
# 6. EXPORT / DOWNLOAD WEIGHTS
# =========================

from IPython.display import FileLink, display
import shutil

best_src  = Path(f"/kaggle/working/runs/{RUN_NAME}/weights/best.pt")
best_copy = Path("/kaggle/working/best.pt")

if best_src.exists():
    shutil.copy(str(best_src), str(best_copy))
    print("Copied best weights to:", best_copy)
    display(FileLink("best.pt"))
else:
    print("best.pt not found yet — has training completed?")

## How to know it worked

Before training starts, the notebook prints dataset counts. With the current dataset you should see something like:



When YOLO starts, look for:



Training will run for up to 100 epochs with early stopping (patience=15). The best checkpoint is saved at:



Run **Cell 6** at any time to copy and download it.